In [3]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 


In [4]:
df = pd.read_csv('data_for_preprocessing.csv')
df.head()

,Unnamed: 0,Text,Author
0,0,This study investigates the chemical compositi...,AI
1,1,This study explores the cultural history of oi...,AI
2,2,Isolation of human peripheral blood mononucle...,Human
3,3,Dynamic Bayesian Networks (DBNs) are probabil...,Human
4,4,"Within volleyball, performance analysis is em...",Human


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6069 entries, 0 to 6068
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  6069 non-null   int64 
 1   Text        6069 non-null   object
 2   Author      6069 non-null   object
dtypes: int64(1), object(2)
memory usage: 142.4+ KB


In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
unique_author = df['Author'].unique()
author_numbers = {}
i = 0
for aut in unique_author:
    author_numbers[aut] = i 
    i+=1

df['Author'] = df['Author'].map(author_numbers)

In [8]:
df.head()

,Unnamed: 0,Text,Author
0,0,This study investigates the chemical compositi...,0
1,1,This study explores the cultural history of oi...,0
2,2,Isolation of human peripheral blood mononucle...,1
3,3,Dynamic Bayesian Networks (DBNs) are probabil...,1
4,4,"Within volleyball, performance analysis is em...",1


In [9]:
df['Text'] = df['Text'].apply(lambda x : x.lower())

In [10]:
import string 
def remove_punc(txt):
    return txt.translate(str.maketrans('','',string.punctuation))
    

In [11]:
df['Text'].apply(remove_punc)

0       this study investigates the chemical compositi...
1       this study explores the cultural history of oi...
2        isolation of human peripheral blood mononucle...
3        dynamic bayesian networks dbns are probabilis...
4        within volleyball performance analysis is emp...
                              ...                        
6064     computational models serve as useful compleme...
6065     during transport and storage of drinking wate...
6066     the parameter values of neural networks will ...
6067     crumb rubber modified asphalt crma offers a v...
6068    generative ai for video is computationally exp...
Name: Text, Length: 6069, dtype: object

In [12]:
def remove_numbers(txt):
    new=""
    for i in txt: 
        if not i.isdigit(): 
            new = new+i
    return new 
df['Text'] = df['Text'].apply(remove_numbers)

In [13]:
import nltk 
from nltk.corpus import stopwords 
from nltk.tokenize import word_tokenize

In [14]:
stop_words = set(stopwords.words('english'))
stop_words

{'a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 "he's",
 'her',
 'here',
 'hers',
 'herself',
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 "i'll",
 "i'm",
 "i've",
 'if',
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [15]:
df.loc[1]['Text']

'this study explores the cultural history of oil wrestling in kırkpınar, analyzing its status as an intangible cultural heritage and its modern organization.'

In [16]:
def remove(txt):
    words = txt.split()
    cleaned = [ ]
    for i in words:
        if not i in stop_words:
            cleaned.append(i)
    return ' '.join(cleaned)

In [17]:
df['Text'] = df['Text'].apply(remove)

In [18]:
df.loc[1]['Text']

'study explores cultural history oil wrestling kırkpınar, analyzing status intangible cultural heritage modern organization.'

In [19]:
df.head()

,Unnamed: 0,Text,Author
0,0,study investigates chemical composition therma...,0
1,1,study explores cultural history oil wrestling ...,0
2,2,isolation human peripheral blood mononuclear c...,1
3,3,dynamic bayesian networks (dbns) probabilistic...,1
4,4,"within volleyball, performance analysis employ...",1


In [20]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df['Text'], df['Author'],test_size=0.20, random_state=42)

In [23]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfidf,y_train)

MultinomialNB()

In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=200),
    "SVM": LinearSVC(),
    "Random Forest": RandomForestClassifier()
}

results = {}

for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    predictions = model.predict(X_test_tfidf)
    accuracy = accuracy_score(y_test, predictions)
    results[name] = accuracy

for model, score in results.items():
    print(model, ":", score)

Naive Bayes : 0.9415156507413509
Logistic Regression : 0.9835255354200988
SVM : 0.9917627677100495
Random Forest : 0.9909390444810544


In [25]:
import pandas as pd

comparison = pd.DataFrame(results.items(), columns=["Model","Accuracy"])
print(comparison)

                 Model  Accuracy
0          Naive Bayes  0.941516
1  Logistic Regression  0.983526
2                  SVM  0.991763
3        Random Forest  0.990939


In [26]:
best_model = max(results, key=results.get)
print("Best Model:", best_model)

Best Model: SVM


In [29]:
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)

model = LogisticRegression(max_iter=200)
model.fit(X_train_vec, y_train)

pickle.dump(model, open("ai_human_model.pkl","wb"))
pickle.dump(vectorizer, open("vectorizer.pkl","wb"))